In [ ]:
import json
import numpy as np
import ollama
import chromadb
from pathlib import Path

cleaned_documents_file = Path("../data/processed/cleaned_documents.json")
chunked_documents_file = Path("../data/processed/chunked_documents.json")
embedding_file = Path("../data/processed/embeddings.npy")
chroma_path = Path("../data/chroma")

print("Setup complete")

In [ ]:
#loading documents

with open(cleaned_documents_file, "r", encoding="utf-8") as f:
    cleaned_documents = json.load(f)

print("Total documents:", len(cleaned_documents))

In [ ]:
#loading chunks 

with open(chunked_documents_file, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print("Total chunks:", len(all_chunks))

In [ ]:

#loading embeddings 

embeddings_array = np.load(embedding_file)

print("Embeddings shape:", embeddings_array.shape)

In [ ]:
#checking if chunks and embeddings are loaded and matches
print("Chunks:", len(all_chunks))
print("Embeddings:", len(embeddings_array))

assert len(all_chunks) == len(embeddings_array)

print("Chunk and embedding counts match")

In [ ]:
#creating a chromadb client to store the embeddings

client = chromadb.PersistentClient(
    path=str(chroma_path)
)

print("ChromaDB client created successfully")

In [ ]:
#creating a collection to store into chromadb

collection = client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)
print("Current count:", collection.count())

In [ ]:
#storing the chunks in batched of size = 100

batch_size = 100

for start in range(0, len(all_chunks), batch_size):

    end = min(start + batch_size, len(all_chunks))

    batch_chunks = all_chunks[start:end]
    batch_embeddings = embeddings_array[start:end]

    collection.add(
        ids=[chunk["chunk_id"] for chunk in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        documents=[chunk["text"] for chunk in batch_chunks],
        metadatas=[
            {
                "doc_id": chunk["doc_id"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            }
            for chunk in batch_chunks
        ]
    )

    print(f"Inserted {end}/{len(all_chunks)} chunks")

In [ ]:
#checking the collection count

print("Chroma collection count:", collection.count())

In [ ]:
#Fetching the first chunk for verifying 

result = collection.get(
    ids=[all_chunks[0]["chunk_id"]],
    include=["embeddings", "documents", "metadatas"]
)

print("ID:", result["ids"][0])
print("Embedding dimensions:", len(result["embeddings"][0]))
print("Document:", result["documents"][0][:300])
print("Metadata:", result["metadatas"][0])